# Langfuse Integration Test Notebook

This notebook verifies the Langfuse observability integration with the e-Commerce Multi-Agent system.

**Configuration:**
- Session ID: `e-Commerce Multi Agents`
- User ID: `001`
- Uses Azure OpenAI via company's DIAL API (bypasses OpenAI key requirement)

**Tests:**
1. Langfuse client connection
2. Manual trace creation
3. LLM call tracing with @observe decorator
4. Full agent workflow tracing

In [9]:
# Cell 1: Setup - Add project root to path and configure logging
import sys
import os
import logging

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print(f"✅ Project root added to path: {project_root}")

✅ Project root added to path: c:\Users\DevavarapuSaiRuthvik\Desktop\E-Commerce_Brain\e_commerce_brain


## Test 1: Verify Langfuse Configuration

Load settings and verify Langfuse credentials are properly configured.

In [10]:
# Cell 2: Verify Langfuse Configuration
from backend.settings import Settings

print("=" * 60)
print("LANGFUSE CONFIGURATION")
print("=" * 60)

# Check if credentials are set
langfuse_public_key = Settings.LANGFUSE_PUBLIC_KEY
langfuse_secret_key = Settings.LANGFUSE_SECRET_KEY
langfuse_base_url = Settings.LANGFUSE_BASE_URL
session_id = Settings.LANGFUSE_SESSION_ID
user_id = Settings.LANGFUSE_USER_ID

print(f"✅ Public Key: {langfuse_public_key[:20]}..." if langfuse_public_key else "❌ Public Key: NOT SET")
print(f"✅ Secret Key: {langfuse_secret_key[:20]}..." if langfuse_secret_key else "❌ Secret Key: NOT SET")
print(f"✅ Base URL: {langfuse_base_url}")
print(f"✅ Session ID: {session_id}")
print(f"✅ User ID: {user_id}")
print("=" * 60)

# Validate all required settings are present
assert langfuse_public_key, "LANGFUSE_PUBLIC_KEY is required"
assert langfuse_secret_key, "LANGFUSE_SECRET_KEY is required"
print("\n✅ All Langfuse configuration validated successfully!")

LANGFUSE CONFIGURATION
✅ Public Key: pk-lf-2b77bbe3-d9d3-...
✅ Secret Key: sk-lf-590dae5f-1649-...
✅ Base URL: https://cloud.langfuse.com
✅ Session ID: e-Commerce Multi Agents
✅ User ID: 001

✅ All Langfuse configuration validated successfully!


## Test 2: Initialize Langfuse Client

Create a Langfuse client instance and verify connection to Langfuse cloud.

In [11]:
# Cell 3: Initialize Langfuse Client
from langfuse import Langfuse

# Create Langfuse client
langfuse_client = Langfuse(
    secret_key=Settings.LANGFUSE_SECRET_KEY,
    public_key=Settings.LANGFUSE_PUBLIC_KEY,
    host=Settings.LANGFUSE_BASE_URL
)

print("✅ Langfuse client initialized successfully!")
print(f"   Host: {Settings.LANGFUSE_BASE_URL}")

✅ Langfuse client initialized successfully!
   Host: https://cloud.langfuse.com


## Test 3: Create a Manual Trace

Create a simple trace manually to verify Langfuse is recording traces correctly.

In [12]:
# Cell 4: Create a Manual Trace
from datetime import datetime

# Start a span which implicitly creates a trace
with langfuse_client.start_as_current_span(
    name="test_manual_trace",
    metadata={
        "test_type": "manual_trace_verification",
        "timestamp": datetime.now().isoformat()
    },
    input={"test_input": "Hello from e-Commerce Brain!"},
    output={"test_output": "Trace created successfully"}
) as span:
    # Update the current trace with session_id and user_id
    langfuse_client.update_current_trace(
        name="test_manual_trace",
        session_id=Settings.LANGFUSE_SESSION_ID,
        user_id=Settings.LANGFUSE_USER_ID
    )
    
    # Get trace ID for display
    trace_id = langfuse_client.get_current_trace_id()

print(f"✅ Manual trace created!")
print(f"   Trace ID: {trace_id}")
print(f"   Session ID: {Settings.LANGFUSE_SESSION_ID}")
print(f"   User ID: {Settings.LANGFUSE_USER_ID}")

# Flush to ensure trace is sent
langfuse_client.flush()
print("✅ Trace flushed to Langfuse cloud")

✅ Manual trace created!
   Trace ID: 4d6f72a40cc3ab40ff5dbade2e20f50f
   Session ID: e-Commerce Multi Agents
   User ID: 001
✅ Trace flushed to Langfuse cloud


## Test 4: Test @observe Decorator with Azure OpenAI

Test the @observe decorator with a simple function that calls Azure OpenAI (your company's DIAL API).

In [13]:
# Cell 5: Test @observe Decorator with Azure OpenAI
from langfuse import observe
from langchain_openai import AzureChatOpenAI

@observe(name="test_azure_llm_call")
def test_llm_with_tracing(prompt: str) -> str:
    """Test function that calls Azure OpenAI with Langfuse tracing."""
    
    # Initialize Azure OpenAI (using company's DIAL API)
    llm = AzureChatOpenAI(
        api_key=Settings.DIAL_API_KEY,
        azure_endpoint=Settings.AZURE_ENDPOINT,
        api_version=Settings.API_VERSION,
        model="gpt-4",
        temperature=0.3
    )
    
    # Make the call
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Respond briefly."},
        {"role": "user", "content": prompt}
    ]
    
    response = llm.invoke(messages)
    result = response.content.strip()
    
    return result

# Test the function
print("🔄 Testing Azure OpenAI LLM call with Langfuse tracing...")
response = test_llm_with_tracing("What is 2 + 2? Answer in one word.")
print(f"✅ LLM Response: {response}")

# Flush traces
langfuse_client.flush()
print("✅ Trace sent to Langfuse cloud")

🔄 Testing Azure OpenAI LLM call with Langfuse tracing...
✅ LLM Response: Four.
✅ Trace sent to Langfuse cloud


## Test 5: Test LLMFormatter with Langfuse Integration

Test the actual LLMFormatter class from the project to ensure it's properly traced.

In [14]:
# Cell 6: Test LLMFormatter with Langfuse Integration
from backend.utils.llm_formatter import LLMFormatter

# Initialize formatter for sales agent
print("🔄 Initializing LLMFormatter for 'sales' agent...")
formatter = LLMFormatter("sales")
print("✅ LLMFormatter initialized with Langfuse tracing")

# Test format_finding method (this will be traced)
print("\n🔄 Testing format_finding method...")
try:
    # Mock data for testing
    raw_metrics = {
        "yesterday_revenue": 8500.00,
        "avg_revenue": 10000.00,
        "yesterday_orders": 85,
        "avg_orders": 100
    }
    
    analysis_results = {
        "revenue_drop_pct": 15.0,
        "order_drop_pct": 15.0,
        "aov_drop_pct": 0.0,
        "drop_cause": "order_volume"
    }
    
    # This call will be traced by Langfuse
    finding = formatter.format_finding(raw_metrics, analysis_results)
    print(f"✅ Finding generated: {finding[:100]}...")
    
    # Flush traces
    langfuse_client.flush()
    print("✅ Trace sent to Langfuse cloud")
    
except Exception as e:
    print(f"⚠️ Error (expected if prompts not available): {e}")

🔄 Initializing LLMFormatter for 'sales' agent...


2026-01-27 17:37:57,772 - backend.utils.llm_formatter - INFO - [LLMFormatter:sales] Initialized with Langfuse tracing
2026-01-27 17:37:57,772 - backend.utils.llm_formatter - INFO - [LLMFormatter:sales] Formatting finding...


✅ LLMFormatter initialized with Langfuse tracing

🔄 Testing format_finding method...


2026-01-27 17:38:00,033 - backend.utils.llm_formatter - INFO - [LLMFormatter:sales] Finding formatted: Sales dropped 15% yesterday (₹8,500 vs ₹10,000 average) due to a 15% decline in order volume (85 ord...


✅ Finding generated: Sales dropped 15% yesterday (₹8,500 vs ₹10,000 average) due to a 15% decline in order volume (85 ord...
✅ Trace sent to Langfuse cloud


## Test 6: Verify Traces in Langfuse Dashboard

After running the tests above, you can verify the traces in the Langfuse dashboard:

1. Go to: https://cloud.langfuse.com
2. Login with your credentials
3. Look for traces with:
   - **Session ID**: `e-Commerce Multi Agents`
   - **User ID**: `001`
   - **Trace names**: `test_manual_trace`, `test_azure_llm_call`, `format_finding`

In [16]:
# Cell 7: Summary - Print all test results
print("=" * 60)
print("LANGFUSE INTEGRATION TEST SUMMARY")
print("=" * 60)
print(f"""
✅ Configuration loaded from .env file
✅ Langfuse client initialized
✅ Manual trace created
✅ @observe decorator working with Azure OpenAI
✅ LLMFormatter traced successfully

📊 Check your traces at: {Settings.LANGFUSE_BASE_URL}

🔍 Filter by:
   - Session ID: {Settings.LANGFUSE_SESSION_ID}
   - User ID: {Settings.LANGFUSE_USER_ID}
""")
print("=" * 60)

LANGFUSE INTEGRATION TEST SUMMARY

✅ Configuration loaded from .env file
✅ Langfuse client initialized
✅ Manual trace created
✅ @observe decorator working with Azure OpenAI
✅ LLMFormatter traced successfully

📊 Check your traces at: https://cloud.langfuse.com

🔍 Filter by:
   - Session ID: e-Commerce Multi Agents
   - User ID: 001

